# Kelvin-wave composites

Workflow: **setup → shared calculations → event selection → field loading → composites → figures (including DCIN)**.

Run from top to bottom for a fresh analysis. All definitions precede their use. After computing `composite`, rerun the figure section to adjust the plots without reloading data or recomputing composites.

The analysis keeps the existing time/zonal-mean baselines, latitude/event averaging, native Q1 grid, and 10° radiative smoothing. Full T and Q are used only to derive potential temperatures before compositing. Five PNG figures are saved to `FIGURE_DIR`; DCIN appears below the Q1–temperature profiles.


## 1. Imports and settings


In [ ]:
import sys

import numpy as np
import netCDF4 as nc

from pathlib import Path
from datetime import datetime
from scipy.ndimage import convolve1d, minimum_filter
from matplotlib import pyplot as plt
from matplotlib.colors import TwoSlopeNorm
sys.path.append("/home/b11209013/Manuscript_Prep/KW_CRI/src")
import spectrum #type: ignore


In [ ]:
START_DATE = "2006-01-01"
END_DATE = "2018-01-01"  # Exclusive upper bound.
LATITUDE_BOUNDS = (-5.0, 5.0)
OLR_PATH = "/work/DATA/Satellite/OLR/olr_anomaly.nc"
ERA5_W_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/w.nc"
ERA5_T_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/T.nc"
ERA5_Q_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/Q.nc"
ERA5_Q1_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/Q1_no_rm3harm.nc"
CLOUDSAT_QR_PATH = "/data92/b11209013/CloudSat/DATA/QR_gridded_15layer_-5_5.nc"

ERA5_T_FULL_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/T_no_rm3harm.nc"
ERA5_Q_FULL_PATH = "/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/Q_no_rm3harm.nc"


FIGURE_DIR = Path("/home/b11209013/Manuscript_Prep/KW_CRI/Figure")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PRESSURE_TICKS = [100, 200, 300, 400, 500, 600, 800, 1000]

# Thermodynamic constants (SI units).
P0_PA = 100000.0
R_D = 287.05
C_P = 1004.5
L_V = 2.5e6

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 14,
        "axes.labelsize": 14,
        "axes.titlesize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "legend.fontsize": 12,
        "figure.titlesize": 18,
    }
)


## 2. Shared calculations

The same `composite_profiles` function serves all fields. Its optional `transform` calculates nonlinear diagnostics on the input chunks before subtracting their baselines. Inputs remain available for rerunning the calculation.


In [ ]:
def convert_time(time_variable: nc.Variable) -> np.ndarray:
    """Return a netCDF time coordinate as YYYY-MM-DD strings."""
    # CDO also stores dates numerically as YYYYMMDD.fraction_of_day.
    if time_variable.units == "day as %Y%m%d.%f":
        return np.array([
            datetime.strptime(str(int(value)), "%Y%m%d").strftime("%Y-%m-%d")
            for value in np.asarray(time_variable[:]).ravel()
        ])
    dates = nc.num2date(
        time_variable[:],
        units=time_variable.units,
        calendar=getattr(time_variable, "calendar", "standard"),
    )
    dates_array = np.asarray(dates).ravel()
    return np.asarray(
        [date.strftime("%Y-%m-%d") for date in dates_array],
        dtype=str,
    )

def as_float_array(values: np.ndarray) -> np.ndarray:
    """Convert a netCDF array to float and replace masked values by NaN."""
    masked_values = np.ma.asarray(values, dtype=np.float32)
    return np.asarray(masked_values.filled(np.nan))


In [ ]:
def composite_profiles(
    fields, event_days, event_lons, longitude, chunk_days=8, transform=None,
):
    """Composite anomalies; optionally derive fields before baseline subtraction.

    transform maps a dict of raw chunks/days to a dict of derived arrays with
    the same grid. This supports nonlinear diagnostics without full-field copies.
    """
    if not fields:
        raise ValueError("Provide at least one field to composite.")
    shape = next(iter(fields.values())).shape
    if len(shape) != 4 or any(field.shape != shape for field in fields.values()):
        raise ValueError("Reload the raw fields: expected matching 4-D arrays.")
    nt, nz, ny, nx = shape
    event_days = np.asarray(event_days)
    event_lons = np.asarray(event_lons, dtype=float)
    longitude = np.asarray(longitude, dtype=float)
    if event_days.ndim != 1 or event_lons.shape != event_days.shape:
        raise ValueError("Event days and longitudes must be matching 1-D arrays.")
    if not np.issubdtype(event_days.dtype, np.integer):
        raise ValueError("Event days must be integer indices.")
    if np.any((event_days < 0) | (event_days >= nt)):
        raise ValueError("Event days fall outside the field's time axis.")
    if longitude.shape != (nx,) or not np.all(np.isfinite(longitude)):
        raise ValueError("Longitude must match the field's longitude axis.")

    def divide_valid(total, count):
        result = np.full(total.shape, np.nan, dtype=np.float64)
        np.divide(total, count, out=result, where=count > 0)
        return result

    def read_fields(key):
        values = {name: as_float_array(field[key]) for name, field in fields.items()}
        return values if transform is None else transform(values)

    # Read each chunk once, deriving theta/theta_e before anomaly subtraction.
    # Float64 accumulation limits roundoff over the long time series.
    baseline_totals, baseline_counts = {}, {}
    for start in range(0, nt, chunk_days):
        blocks = read_fields(slice(start, start + chunk_days))
        for name, block in blocks.items():
            if name not in baseline_totals:
                baseline_totals[name] = np.zeros((nz, ny), dtype=np.float64)
                baseline_counts[name] = np.zeros((nz, ny), dtype=np.int64)
            valid = ~np.isnan(block)
            baseline_totals[name] += np.sum(block, axis=(0, 3), where=valid, dtype=np.float64)
            baseline_counts[name] += np.sum(valid, axis=(0, 3), dtype=np.int64)
    baselines = {
        name: divide_valid(total, baseline_counts[name])[..., None]
        for name, total in baseline_totals.items()
    }
    del blocks, block, valid

    # Compute each longitude match once; circular distance handles 0/360 degrees.
    centers = np.array([
        np.argmin(np.abs((longitude - value + 180.0) % 360.0 - 180.0))
        for value in event_lons
    ])
    # Group events by day so multiple centers share the same profile calculation.
    order = np.argsort(event_days, kind="stable")
    unique_days, starts = np.unique(event_days[order], return_index=True)
    ends = np.r_[starts[1:], event_days.size]
    products = {
        f"{name}_gen": name
        for name in ("lw", "sw") if name in baselines and "t" in baselines
    }
    names = (*baselines, *products)
    totals = {name: np.zeros((nz, nx), dtype=np.float64) for name in names}
    counts = {name: np.zeros((nz, nx), dtype=np.int64) for name in names}
    offsets = np.arange(nx) - nx // 2

    for day, start, end in zip(unique_days, starts, ends):
        anomalies = {
            name: values - baselines[name]
            for name, values in read_fields(day).items()
        }
        # Multiply before latitude averaging, retaining the original statistic.
        for product, name in products.items():
            anomalies[product] = anomalies[name] * anomalies["t"]
        for name, values in anomalies.items():
            valid = ~np.isnan(values)
            profile = divide_valid(
                np.sum(values, axis=1, where=valid, dtype=np.float64),
                np.sum(valid, axis=1, dtype=np.int64),
            )
            present = ~np.isnan(profile)
            filled = np.where(present, profile, 0.0)
            for event in order[start:end]:
                indices = (centers[event] + offsets) % nx
                totals[name] += filled[:, indices]
                counts[name] += present[:, indices]

    return {name: divide_valid(totals[name], counts[name]) for name in names}


### Potential-temperature definitions

Using full temperature $T$ in K, pressure $p$ in Pa, and specific humidity $q$ in kg/kg:

$$\theta = T\left(\frac{p_0}{p}\right)^{R_d/C_p},\qquad
r = \frac{q}{1-q},\qquad
\theta_e = \theta\exp\left(\frac{L_v r}{C_p T}\right).$$

Here $p_0=100000$ Pa, $R_d=287.05$ J kg$^{-1}$ K$^{-1}$, $C_p=1004.5$ J kg$^{-1}$ K$^{-1}$, and $L_v=2.5\times10^6$ J kg$^{-1}$. We use the requested simplified expression for $\theta_e$.

The existing `T.nc` and `Q.nc` contain anomalies, so this calculation reads `T_no_rm3harm.nc` and `Q_no_rm3harm.nc`. Both temperatures are calculated at each grid point **before** subtracting each derived field's time/zonal mean and averaging over latitude and events. No seasonal harmonics are removed in this section. The figure shows composite anomalies in K.

References: [potential temperature](https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.potential_temperature.html) and [specific humidity to mixing ratio](https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.mixing_ratio_from_specific_humidity.html).


In [ ]:
def potential_temperatures(temperature_k, specific_humidity, pressure_pa):
    """Return theta and the requested approximate theta_e in K.

    Inputs must be full fields, not anomalies. Pressure must broadcast with T.
    Mask missing/nonphysical values rather than using them in the exponential.
    """
    temperature = np.asarray(np.ma.filled(
        np.ma.asarray(temperature_k, dtype=np.float64), np.nan,
    ))
    humidity = np.asarray(np.ma.filled(
        np.ma.asarray(specific_humidity, dtype=np.float64), np.nan,
    ))
    pressure = np.asarray(pressure_pa, dtype=np.float64)
    if np.any(~np.isfinite(pressure) | (pressure <= 0)):
        raise ValueError("Pressure must be finite and positive, in Pa.")
    temperature = np.where(np.isfinite(temperature) & (temperature > 0), temperature, np.nan)
    humidity = np.where(
        np.isfinite(humidity) & (humidity >= 0) & (humidity < 1), humidity, np.nan,
    )

    # 1. Convert full temperature to potential temperature.
    theta = temperature * (P0_PA / pressure) ** (R_D / C_P)
    # 2. Convert ERA5 specific humidity to water-vapor mixing ratio.
    mixing_ratio = humidity / (1.0 - humidity)
    # 3. Calculate theta_e locally, before any composite averaging.
    theta_e = theta * np.exp(L_V * mixing_ratio / (C_P * temperature))
    return {"theta": theta, "theta_e": theta_e}


## 3. Select Kelvin-wave events


In [ ]:
with nc.Dataset(OLR_PATH, "r") as olr_ds:
    all_olr_dates: np.ndarray = convert_time(olr_ds.variables["time"])
    olr_lat  : np.ndarray = as_float_array(olr_ds.variables["lat"][...])
    olr_lon  : np.ndarray = as_float_array(olr_ds.variables["lon"][...])

    time_mask = (all_olr_dates >= START_DATE) & (all_olr_dates < END_DATE)
    lat_mask  = (olr_lat >= LATITUDE_BOUNDS[0]) & (olr_lat <= LATITUDE_BOUNDS[1])

    olr      : np.ndarray = as_float_array(
        olr_ds.variables["olr"][
            time_mask, lat_mask, :
        ]
    )
    olr_dates: np.ndarray = all_olr_dates[time_mask]

olr_nt, olr_ny, olr_nx = olr.shape


In [ ]:
# remove time and zonal mean
olr_anom: np.ndarray = olr - np.nanmean(olr, axis=(0, -1), keepdims=True)

# symmetrizing data
olr_symm: np.ndarray = spectrum.symm_asym(olr_anom, lat_axis=1)[0].mean(axis=1)
# set wavenumber and frequency
wnum: np.ndarray = np.fft.fftfreq(olr_nx, d=1/olr_nx)
freq: np.ndarray = np.fft.fftfreq(olr_nt, d=1)

wnums, freqs = np.meshgrid(wnum, freq)

# FFT transform
olr_fft: np.ndarray = np.fft.fft(np.fft.ifft(olr_symm, axis=0), axis=1)

# apply masking
def kelvin_disprel(
        edep: float,
        wnum: np.ndarray
) -> np.ndarray:

    return np.sqrt(9.81 * edep) * 86400.0 / (2*np.pi*6.371e6) * wnum

wnum_lim = ((wnums >= 3) & (wnums <= 8)) | ((wnums <= -3) & (wnums >= -8))
freq_lim = ((freqs >= 1/20) & (freqs <= 1/2.5)) | ((freqs <= -1/20) & (freqs >= -1/2.5))
edep_lim = (freqs >= kelvin_disprel(8, wnums)) & (freqs <= kelvin_disprel(90, wnums)) | \
    (freqs <= kelvin_disprel(8, wnums)) & (freqs >= kelvin_disprel(90, wnums))

mask = wnum_lim & freq_lim & edep_lim

olr_mask = np.where(mask, olr_fft, 0.0)

# IFFT transform
Kelvin_recon: np.ndarray = np.fft.ifft(np.fft.fft(olr_mask, axis=0), axis=1).real


In [ ]:
# Calculate statistics
mean: float = Kelvin_recon.flatten().mean()
std : float = Kelvin_recon.flatten().std()

threshold: float = mean - 2.96 * std

# Periodic neighborhood in longitude; adjust zonal size as appropriate
local_minimum = (
    Kelvin_recon
    == minimum_filter(
        Kelvin_recon,
        size=(1, 81),       # 81 longitude grid points, represents 50 degree
        mode=("nearest", "wrap"), #type: ignore
    )
)

selected = local_minimum & (Kelvin_recon <= threshold)
time_idx, lon_idx = np.where(selected)

lon_val: np.ndarray = np.array([
    olr_lon[l] for l in lon_idx
])


## 4. Load composite fields

ERA5 fields use `era5_coords` and the same time indexing. Q1 retains its own coordinates and date matching because its grid differs. All fields are loaded directly as arrays.


In [ ]:
with nc.Dataset(ERA5_W_PATH, "r") as w_ds:

    era5_coords: dict[str, np.ndarray] = {
        key: w_ds.variables[key][...]
        for key in w_ds.dimensions.keys()
    }

    lat_lim = (era5_coords["lat"] >= LATITUDE_BOUNDS[0]) & (era5_coords["lat"] <= LATITUDE_BOUNDS[1])

    w: np.ndarray = w_ds.variables["w"][..., lat_lim, :]

with nc.Dataset(ERA5_Q_PATH, "r") as q_ds:
    q: np.ndarray = q_ds.variables["Q"][..., lat_lim, :]

with nc.Dataset(ERA5_T_PATH, "r") as t_ds:
    t: np.ndarray = t_ds.variables["T"][..., lat_lim, :]


In [ ]:
with nc.Dataset(ERA5_Q1_PATH, "r") as q1_ds:

    q1_coords: dict[str, np.ndarray] = {
        key: as_float_array(q1_ds.variables[key][...])
        for key in q1_ds.dimensions.keys()
    }
    # Q1 has its own coarser grid and descending pressure coordinate.
    q1_lat_lim = (
        (q1_coords["lat"] >= LATITUDE_BOUNDS[0])
        & (q1_coords["lat"] <= LATITUDE_BOUNDS[1])
    )
    q1_dates = convert_time(q1_ds.variables["time"])
    q1_time_lim = (q1_dates >= START_DATE) & (q1_dates < END_DATE)
    q1_dates = q1_dates[q1_time_lim]
    q1: np.ndarray = as_float_array(
        q1_ds.variables["Q1"][q1_time_lim, :, q1_lat_lim, :]
    )

with nc.Dataset(CLOUDSAT_QR_PATH, "r") as cs_ds:
    lw: np.ndarray = cs_ds.variables["QLW"][...]
    sw: np.ndarray = cs_ds.variables["QSW"][...]


In [ ]:
# Load full T and Q arrays just like the earlier ERA5 fields.
# Reuse the same latitude selection, coordinates, and time indexing.
with nc.Dataset(ERA5_T_FULL_PATH) as full_t_ds:
    if full_t_ds["T"].units != "K" or full_t_ds["plev"].units != "Pa":
        raise ValueError("Expected temperature in K and pressure in Pa.")
    t_full: np.ndarray = full_t_ds.variables["T"][..., lat_lim, :]

with nc.Dataset(ERA5_Q_FULL_PATH) as full_q_ds:
    if full_q_ds["Q"].units not in ("kg kg**-1", "kg kg-1", "kg/kg", "1"):
        raise ValueError("Expected specific humidity in kg/kg.")
    q_full: np.ndarray = full_q_ds.variables["Q"][..., lat_lim, :]


## 5. Calculate composites

Base fields and radiative products use the existing ERA5 event indices. Q1 uses the same event dates on its native grid. Potential temperatures use full T and Q, with no additional seasonal-harmonic removal.


In [ ]:
# Keep the all-loaded-day/zonal baseline; only process event days afterward.
# Inputs are the unchanged raw 4-D arrays from the loading cell.
composite = composite_profiles(
    {"w": w, "t": t, "q": q, "lw": lw, "sw": sw},
    time_idx, lon_val, era5_coords["lon"],
)
# Match by date rather than assuming Q1 shares ERA5's time indices.
q1_day_lookup = {date: day for day, date in enumerate(q1_dates)}
event_dates = olr_dates[time_idx]
missing_q1_dates = sorted(set(event_dates) - q1_day_lookup.keys())
if missing_q1_dates:
    raise ValueError(f"Q1 is missing event dates: {missing_q1_dates[:5]}")
q1_event_days = np.array([q1_day_lookup[date] for date in event_dates], dtype=int)
composite["q1"] = composite_profiles(
    {"q1": q1}, q1_event_days, lon_val, q1_coords["lon"],
)["q1"]


In [ ]:
thermo_composite = composite_profiles(
    {"temperature": t_full, "humidity": q_full},
    time_idx, lon_val, era5_coords["lon"],
    transform=lambda values: potential_temperatures(
        values["temperature"], values["humidity"], era5_coords["plev"][:, None, None],
    ),
)

composite.update(thermo_composite)
print(f"Computed theta and theta_e composites for {len(time_idx)} events.")


## 6. Figures

The plotting cells share coordinates and style settings. The first three figures retain their original shading, contours, and pressure scales. Q1 shading is converted from J kg⁻¹ s⁻¹ to K/day; potential-temperature anomalies are shown in K.


In [ ]:
# ERA5 axes are shared by temperature, humidity, and potential temperatures.
NLON = era5_coords["lon"].size
x_rel = (np.arange(NLON) - NLON // 2) * 0.25
plev = era5_coords["plev"] / 100.0

# Q1 has a coarser longitude grid and a separate pressure ordering.
q1_nlon = q1_coords["lon"].size
q1_dlon = float(np.diff(q1_coords["lon"])[0])
q1_x_rel = (np.arange(q1_nlon) - q1_nlon // 2) * q1_dlon
q1_plev = q1_coords["levPa"] / 100.0

# Smooth the radiative-heating composites over 10 degrees longitude.
smoothing_kernel = np.ones(40) / 40
lw_smoothed = convolve1d(composite["lw"], smoothing_kernel, axis=1, mode="wrap")
sw_smoothed = convolve1d(composite["sw"], smoothing_kernel, axis=1, mode="wrap")

lw_gen_smoothed = convolve1d(composite["lw_gen"], smoothing_kernel, axis=1, mode="wrap")
sw_gen_smoothed = convolve1d(composite["sw_gen"], smoothing_kernel, axis=1, mode="wrap")


### DCIN for the lower Q1–temperature panel

Retain the current diagnostic: integrate the composite $\theta_e$ anomaly over 800–1000 hPa, then subtract that integral from its value at 700 hPa.


In [ ]:
# boundary layer moist entropy
bl_lim = (plev >= 800.0) & (plev <= 1000.0)
s_bl: np.ndarray = np.trapz(composite["theta_e"][bl_lim, :], x =plev[bl_lim], axis=0)

# Value at 700 hPa.
s_t = composite["theta_e"][plev==700.0]

# calculate DCIN
DCIN = s_t - s_bl


In [ ]:
def plot_w_panels(contour_fields, titles, filename, log_pressure=False):
    """Draw the shared two-panel vertical-velocity figures."""
    fig, axes = plt.subplots(2, 1, figsize=(11, 10), sharex="col")
    contours = []
    for i, (ax, field, title) in enumerate(zip(axes, contour_fields, titles)):
        shading = ax.contourf(
            x_rel, plev, composite["w"] * 1e2,
            cmap="RdBu_r", norm=TwoSlopeNorm(0.0), alpha=0.4,
        )
        contours.append(ax.contour(x_rel, plev, field, colors="k"))
        ax.minorticks_on()
        if log_pressure:
            ax.set_yscale("log")
        ax.set_yticks(PRESSURE_TICKS)
        ax.set_yticklabels([str(value) for value in PRESSURE_TICKS])
        ax.set_xlim(-50, 50)
        ax.set_ylim(1000, 100)
        ax.set_ylabel("Pressure Level")
        ax.set_title(title)
        ax.grid()
        if i == 0:
            plt.clabel(contours[0], inline=True)

    axes[-1].set_xlabel(r"Relative Longitude [$^\circ$]")
    fig.colorbar(shading, ax=axes, label=r"Vertical velocity [cm s$^{-1}$]")
    plt.clabel(contours[-1], inline=True)
    fig.savefig(FIGURE_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
plot_w_panels(
    [composite["t"], composite["q"] * 1e3],
    [r"$w$ (shading) vs. $T$ (contour; K)",
     r"$w$ (shading) vs. $q$ (contour; g/kg)"],
    "wTQ_composite.png", log_pressure=True,
)


In [ ]:
plot_w_panels(
    [lw_smoothed, sw_smoothed],
    [r"$w$ (shading) vs. $LW$ (contour; K/day)",
     r"$w$ (shading) vs. $SW$ (contour; K/day)"],
    "LWSW_composite.png",
)

plot_w_panels(
    [lw_gen_smoothed, sw_gen_smoothed],
    [r"$w$ (shading) vs. $LW$ (contour; K/day)",
     r"$w$ (shading) vs. $SW$ (contour; K/day)"],
    "LWSW_gen_composite.png",
)


In [ ]:
# Symmetric shading limits based on the displayed longitude range.
q1_visible = composite["q1"][:, np.abs(q1_x_rel) <= 50]
q1_finite = q1_visible[np.isfinite(q1_visible)] * 86400.0/1004.5
if not q1_finite.size:
    raise ValueError("No finite Q1 composite values in the plotted region.")
q1_limit = float(np.max(np.abs(q1_finite))) or 1.0
q1_levels = np.linspace(-q1_limit, q1_limit, 21)

fig = plt.figure(figsize=(11, 8))
# A separate colorbar column keeps the profile and DCIN longitude axes aligned.
grid = fig.add_gridspec(
    2, 2, height_ratios=[3, 1], width_ratios=[1, 0.035],
    hspace=0.12, wspace=0.08,
)
ax = fig.add_subplot(grid[0, 0])
dcin_ax = fig.add_subplot(grid[1, 0], sharex=ax)
cbar_ax = fig.add_subplot(grid[0, 1])
q1_ctf = ax.contourf(
    q1_x_rel, q1_plev, composite["q1"]*86400.0/1004.5,
    levels=q1_levels, cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-q1_limit, vcenter=0.0, vmax=q1_limit),
    extend="both",
)
t_ct = ax.contour(
    x_rel, plev, composite["t"],
    colors="k", linewidths=0.9,
)
ax.clabel(t_ct, inline=True, fmt="%.2f", fontsize=10)
ax.minorticks_on()
ax.set_yscale("log")
ax.set_yticks(PRESSURE_TICKS)
ax.set_yticklabels([str(value) for value in PRESSURE_TICKS])
ax.set_xlim(-50, 50)
ax.set_ylim(1000, 100)
ax.tick_params(axis="x", labelbottom=False)
ax.set_ylabel("Pressure [hPa]")
ax.set_title(r"$Q_1$ anomaly (shading) vs. $T$ anomaly (contour; K)")
ax.grid(alpha=0.3)
fig.colorbar(q1_ctf, cax=cbar_ax, label="Q1 anomaly [K / day]")

# Lower panel: the existing DCIN diagnostic on the same longitude axis.
dcin_ax.plot(x_rel, DCIN.squeeze(), color="k", linewidth=1.5)
dcin_ax.set_xlim(-50, 50)
dcin_ax.set_xlabel(r"Relative Longitude [$^\circ$]")
dcin_ax.set_ylabel("DCIN")
dcin_ax.grid(alpha=0.3)

q1_figure_path = FIGURE_DIR / "Q1T_composite.png"
fig.savefig(q1_figure_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)


In [ ]:
# Reuse the relative longitude and pressure axes from the ERA5 composite plots.
thermo_visible = np.abs(x_rel) <= 50

theta_e_visible = composite["theta_e"][:, thermo_visible]
theta_e_finite = theta_e_visible[np.isfinite(theta_e_visible)]
if not theta_e_finite.size:
    raise ValueError("No finite theta_e composite values in the plotted region.")
theta_e_limit = float(np.max(np.abs(theta_e_finite))) or 1.0

fig, ax = plt.subplots(figsize=(11, 5))
theta_e_ctf = ax.contourf(
    x_rel, plev, composite["theta_e"],
    levels=np.linspace(-theta_e_limit, theta_e_limit, 21),
    cmap="RdBu_r",
    norm=TwoSlopeNorm(vmin=-theta_e_limit, vcenter=0.0, vmax=theta_e_limit),
    extend="both",
)
theta_ct = ax.contour(
    x_rel, plev, composite["theta"],
    colors="k", linewidths=0.9,
)
ax.clabel(theta_ct, inline=True, fmt="%.1f", fontsize=10)
ax.set_yscale("log")
ax.set_yticks(PRESSURE_TICKS)
ax.set_yticklabels([str(value) for value in PRESSURE_TICKS])
ax.set_ylim(1000, 100)
ax.set_xlim(-50, 50)
ax.set_xlabel(r"Relative Longitude [$^\circ$]")
ax.set_ylabel("Pressure [hPa]")
ax.set_title(r"$\theta_e$ anomaly (shading) vs. $\theta$ anomaly (contour; K)")
ax.grid(alpha=0.3)
fig.colorbar(theta_e_ctf, ax=ax, label=r"Equivalent potential temperature anomaly [K]")

theta_e_figure_path = FIGURE_DIR / "theta_e_composite.png"
fig.savefig(theta_e_figure_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
